In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Banking Analytics').getOrCreate()

---

# Banking Transactions Analytics

## Step 1 — Create Data in Colab

```
customers_csv = """customer_id,name,city,age,account_type,signup_date
1,Rahul,Hyderabad,29,Savings,2023-01-10
2,Sneha,Bangalore,32,Current,2023-02-12
3,Arjun,Mumbai,27,Savings,2023-03-14
4,Priya,Delhi,35,Savings,2023-04-15
5,Karan,Chennai,30,Current,2023-05-11
6,Meera,Hyderabad,31,Savings,2023-06-10
7,Amit,Pune,38,Current,2023-06-22
8,Neha,Delhi,26,Savings,2023-07-10
9,Divya,Bangalore,28,Savings,2023-07-15
10,Vikram,Mumbai,42,Current,2023-08-01
11,Farhan,Hyderabad,34,Savings,2023-08-10
12,Simran,Delhi,25,Savings,2023-08-21
"""
```

```
accounts_csv = """account_id,customer_id,branch,balance
1001,1,Hyderabad Main,85000
1002,2,Bangalore Central,120000
1003,3,Mumbai West,45000
1004,4,Delhi North,95000
1005,5,Chennai South,60000
1006,6,Hyderabad Main,150000
1007,7,Pune East,30000
1008,8,Delhi North,70000
1009,9,Bangalore Central,110000
1010,10,Mumbai West,200000
1011,11,Hyderabad Main,50000
1012,12,Delhi North,40000
"""
```

```
transactions_csv = """txn_id,account_id,txn_type,amount,txn_date
1,1001,Credit,25000,2024-03-01
2,1002,Debit,15000,2024-03-01
3,1003,Credit,10000,2024-03-02
4,1004,Debit,5000,2024-03-02
5,1005,Credit,30000,2024-03-03
6,1006,Debit,20000,2024-03-03
7,1007,Credit,8000,2024-03-04
8,1008,Debit,12000,2024-03-04
9,1009,Credit,40000,2024-03-05
10,1010,Debit,35000,2024-03-05
11,1001,Debit,7000,2024-03-06
12,1002,Credit,18000,2024-03-06
13,1006,Credit,50000,2024-03-07
14,1010,Credit,60000,2024-03-07
15,1011,Debit,9000,2024-03-08
16,1012,Credit,16000,2024-03-08
17,1003,Debit,4000,2024-03-09
18,1004,Credit,22000,2024-03-09
19,1005,Debit,11000,2024-03-10
20,1009,Debit,14000,2024-03-10
"""
```

```
branches_csv = """branch,region,manager
Hyderabad Main,South,Ramesh
Bangalore Central,South,Leena
Mumbai West,West,Joseph
Delhi North,North,Sara
Chennai South,South,Kumar
Pune East,West,Anita
"""
```

```
logs_txt = """Rahul login
Sneha login
Rahul transfer
Arjun login
Priya withdrawal
Rahul logout
Meera login
Vikram transfer
Divya login
Farhan login
Simran withdrawal
Neha login
Amit deposit
Karan login
Meera transfer
Vikram login
Rahul deposit
Sneha withdrawal
Farhan transfer
Divya logout
"""
```

```
customer_profiles_json = """[
 {
 "customer_id": 1,
 "name": "Rahul",
 "contact": {"email": "rahul@mail.com", "phone": "9000011111"},
 "services": ["UPI", "Credit Card", "Net Banking"]
 },
 {
 "customer_id": 2,
 "name": "Sneha",
 "contact": {"email": "sneha@mail.com", "phone": "9000022222"},
 "services": ["UPI", "Debit Card"]
 },
 {
 "customer_id": 3,
 "name": "Arjun",
 "contact": {"email": "arjun@mail.com", "phone": "9000033333"},
 "services": ["Net Banking", "Loan"]
 },
 {
 "customer_id": 6,
 "name": "Meera",
 "contact": {"email": "meera@mail.com", "phone": "9000066666"},
 "services": ["UPI", "Credit Card", "Loan"]
 },
 {
 "customer_id": 10,
 "name": "Vikram",
 "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
 "services": ["Net Banking", "Wealth"]
 }
]
"""
```

```
with open("bank_customers.csv", "w") as f:
 f.write(customers_csv)

with open("accounts.csv", "w") as f:
 f.write(accounts_csv)

with open("transactions.csv", "w") as f:
 f.write(transactions_csv)

with open("branches.csv", "w") as f:
 f.write(branches_csv)

with open("bank_logs.txt", "w") as f:
 f.write(logs_txt)

with open("customer_profiles.json", "w") as f:
 f.write(customer_profiles_json)

print("Banking datasets created")
```


In [2]:
customers_csv = """customer_id,name,city,age,account_type,signup_date
1,Rahul,Hyderabad,29,Savings,2023-01-10
2,Sneha,Bangalore,32,Current,2023-02-12
3,Arjun,Mumbai,27,Savings,2023-03-14
4,Priya,Delhi,35,Savings,2023-04-15
5,Karan,Chennai,30,Current,2023-05-11
6,Meera,Hyderabad,31,Savings,2023-06-10
7,Amit,Pune,38,Current,2023-06-22
8,Neha,Delhi,26,Savings,2023-07-10
9,Divya,Bangalore,28,Savings,2023-07-15
10,Vikram,Mumbai,42,Current,2023-08-01
11,Farhan,Hyderabad,34,Savings,2023-08-10
12,Simran,Delhi,25,Savings,2023-08-21
"""

accounts_csv = """account_id,customer_id,branch,balance
1001,1,Hyderabad Main,85000
1002,2,Bangalore Central,120000
1003,3,Mumbai West,45000
1004,4,Delhi North,95000
1005,5,Chennai South,60000
1006,6,Hyderabad Main,150000
1007,7,Pune East,30000
1008,8,Delhi North,70000
1009,9,Bangalore Central,110000
1010,10,Mumbai West,200000
1011,11,Hyderabad Main,50000
1012,12,Delhi North,40000
"""

transactions_csv = """txn_id,account_id,txn_type,amount,txn_date
1,1001,Credit,25000,2024-03-01
2,1002,Debit,15000,2024-03-01
3,1003,Credit,10000,2024-03-02
4,1004,Debit,5000,2024-03-02
5,1005,Credit,30000,2024-03-03
6,1006,Debit,20000,2024-03-03
7,1007,Credit,8000,2024-03-04
8,1008,Debit,12000,2024-03-04
9,1009,Credit,40000,2024-03-05
10,1010,Debit,35000,2024-03-05
11,1001,Debit,7000,2024-03-06
12,1002,Credit,18000,2024-03-06
13,1006,Credit,50000,2024-03-07
14,1010,Credit,60000,2024-03-07
15,1011,Debit,9000,2024-03-08
16,1012,Credit,16000,2024-03-08
17,1003,Debit,4000,2024-03-09
18,1004,Credit,22000,2024-03-09
19,1005,Debit,11000,2024-03-10
20,1009,Debit,14000,2024-03-10
"""

branches_csv = """branch,region,manager
Hyderabad Main,South,Ramesh
Bangalore Central,South,Leena
Mumbai West,West,Joseph
Delhi North,North,Sara
Chennai South,South,Kumar
Pune East,West,Anita
"""

logs_txt = """Rahul login
Sneha login
Rahul transfer
Arjun login
Priya withdrawal
Rahul logout
Meera login
Vikram transfer
Divya login
Farhan login
Simran withdrawal
Neha login
Amit deposit
Karan login
Meera transfer
Vikram login
Rahul deposit
Sneha withdrawal
Farhan transfer
Divya logout
"""

customer_profiles_json = """[
  {
    "customer_id": 1,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000011111"},
    "services": ["UPI", "Credit Card", "Net Banking"]
  },
  {
    "customer_id": 2,
    "name": "Sneha",
    "contact": {"email": "sneha@mail.com", "phone": "9000022222"},
    "services": ["UPI", "Debit Card"]
  },
  {
    "customer_id": 3,
    "name": "Arjun",
    "contact": {"email": "arjun@mail.com", "phone": "9000033333"},
    "services": ["Net Banking", "Loan"]
  },
  {
    "customer_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "services": ["UPI", "Credit Card", "Loan"]
  },
  {
    "customer_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "services": ["Net Banking", "Wealth"]
  }
]
"""

with open("bank_customers.csv", "w") as f:
    f.write(customers_csv)

with open("accounts.csv", "w") as f:
    f.write(accounts_csv)

with open("transactions.csv", "w") as f:
    f.write(transactions_csv)

with open("branches.csv", "w") as f:
    f.write(branches_csv)

with open("bank_logs.txt", "w") as f:
    f.write(logs_txt)

with open("customer_profiles.json", "w") as f:
    f.write(customer_profiles_json)

print("Banking datasets created")

Banking datasets created


---

## Step 2 — Load the Data

```
customers = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)
```

---

In [3]:
customers = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)

---
# 100 PySpark Exercises
### Section 1 — DataFrame Basics

1. Show all customers.
2. Show all accounts.
3. Show all transactions.
4. Show all branches.
5. Print schema of customers.
6. Print schema of transactions.
7. Count total customers.
8. Count total accounts.
9. Count total transactions.
10. Display first 5 transactions.
---

In [4]:
customers.show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [5]:
accounts.show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1001|          1|   Hyderabad Main|  85000|
|      1002|          2|Bangalore Central| 120000|
|      1003|          3|      Mumbai West|  45000|
|      1004|          4|      Delhi North|  95000|
|      1005|          5|    Chennai South|  60000|
|      1006|          6|   Hyderabad Main| 150000|
|      1007|          7|        Pune East|  30000|
|      1008|          8|      Delhi North|  70000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
|      1011|         11|   Hyderabad Main|  50000|
|      1012|         12|      Delhi North|  40000|
+----------+-----------+-----------------+-------+



In [6]:
transactions.show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [7]:
branches.show()

+-----------------+------+-------+
|           branch|region|manager|
+-----------------+------+-------+
|   Hyderabad Main| South| Ramesh|
|Bangalore Central| South|  Leena|
|      Mumbai West|  West| Joseph|
|      Delhi North| North|   Sara|
|    Chennai South| South|  Kumar|
|        Pune East|  West|  Anita|
+-----------------+------+-------+



In [8]:
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [9]:
transactions.printSchema()

root
 |-- txn_id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- txn_date: date (nullable = true)



In [10]:
customers.count()

12

In [11]:
accounts.count()

12

In [12]:
transactions.count()

20

In [13]:
transactions.show(5)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 5 rows


---

## Section 2 — Select, Rename, Filter

11. Select customer name, city, and account type.
12. Select account_id and balance.
13. Rename balance to current_balance.
14. Rename txn_type to transaction_type.
15. Show customers from Hyderabad.
16. Show customers older than 30.
17. Show Savings account customers.
18. Show accounts with balance greater than 100000.
19. Show Credit transactions.
20. Show transactions with amount greater than 20000.

---


In [14]:
customers.select(customers.name,customers.city,customers.account_type).show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+



In [15]:
accounts.select(accounts.account_id,accounts.balance).show()

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+



In [18]:
accounts=accounts.withColumnRenamed("account_id","account_number")
accounts.show()

+--------------+-----------+-----------------+-------+
|account_number|customer_id|           branch|balance|
+--------------+-----------+-----------------+-------+
|          1001|          1|   Hyderabad Main|  85000|
|          1002|          2|Bangalore Central| 120000|
|          1003|          3|      Mumbai West|  45000|
|          1004|          4|      Delhi North|  95000|
|          1005|          5|    Chennai South|  60000|
|          1006|          6|   Hyderabad Main| 150000|
|          1007|          7|        Pune East|  30000|
|          1008|          8|      Delhi North|  70000|
|          1009|          9|Bangalore Central| 110000|
|          1010|         10|      Mumbai West| 200000|
|          1011|         11|   Hyderabad Main|  50000|
|          1012|         12|      Delhi North|  40000|
+--------------+-----------+-----------------+-------+



In [19]:
transactions=transactions.withColumnRenamed("txn_id","transaction_id")
transactions.show()

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|             1|      1001|  Credit| 25000|2024-03-01|
|             2|      1002|   Debit| 15000|2024-03-01|
|             3|      1003|  Credit| 10000|2024-03-02|
|             4|      1004|   Debit|  5000|2024-03-02|
|             5|      1005|  Credit| 30000|2024-03-03|
|             6|      1006|   Debit| 20000|2024-03-03|
|             7|      1007|  Credit|  8000|2024-03-04|
|             8|      1008|   Debit| 12000|2024-03-04|
|             9|      1009|  Credit| 40000|2024-03-05|
|            10|      1010|   Debit| 35000|2024-03-05|
|            11|      1001|   Debit|  7000|2024-03-06|
|            12|      1002|  Credit| 18000|2024-03-06|
|            13|      1006|  Credit| 50000|2024-03-07|
|            14|      1010|  Credit| 60000|2024-03-07|
|            15|      1011|   Debit|  9000|2024-03-08|
|         

In [20]:
customers.filter(customers.city=="Hyderabad").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [21]:
customers.filter(customers.age>30).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [22]:
customers.filter(customers.account_type=="Savings").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [23]:
accounts.filter(accounts.balance>100000).show()

+--------------+-----------+-----------------+-------+
|account_number|customer_id|           branch|balance|
+--------------+-----------+-----------------+-------+
|          1002|          2|Bangalore Central| 120000|
|          1006|          6|   Hyderabad Main| 150000|
|          1009|          9|Bangalore Central| 110000|
|          1010|         10|      Mumbai West| 200000|
+--------------+-----------+-----------------+-------+



In [24]:
transactions.filter(transactions.txn_type=="Credit").show()

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|             1|      1001|  Credit| 25000|2024-03-01|
|             3|      1003|  Credit| 10000|2024-03-02|
|             5|      1005|  Credit| 30000|2024-03-03|
|             7|      1007|  Credit|  8000|2024-03-04|
|             9|      1009|  Credit| 40000|2024-03-05|
|            12|      1002|  Credit| 18000|2024-03-06|
|            13|      1006|  Credit| 50000|2024-03-07|
|            14|      1010|  Credit| 60000|2024-03-07|
|            16|      1012|  Credit| 16000|2024-03-08|
|            18|      1004|  Credit| 22000|2024-03-09|
+--------------+----------+--------+------+----------+



In [25]:
transactions.filter(transactions.amount>20000).show()

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|             1|      1001|  Credit| 25000|2024-03-01|
|             5|      1005|  Credit| 30000|2024-03-03|
|             9|      1009|  Credit| 40000|2024-03-05|
|            10|      1010|   Debit| 35000|2024-03-05|
|            13|      1006|  Credit| 50000|2024-03-07|
|            14|      1010|  Credit| 60000|2024-03-07|
|            18|      1004|  Credit| 22000|2024-03-09|
+--------------+----------+--------+------+----------+



---

## Section 3 — Sorting and Limit

21. Sort customers by age ascending.
22. Sort customers by age descending.
23. Sort accounts by balance descending.
24. Show top 5 highest balance accounts.
25. Show bottom 5 lowest balance accounts.
26. Sort transactions by amount descending.
27. Show top 3 biggest transactions.
28. Sort transactions by txn_date.
29. Sort customers by city and age.
30. Show latest 5 transactions.

---

In [26]:
customers.sort(customers.age).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+



In [27]:
customers.sort(customers.age.desc()).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [28]:
accounts.sort(accounts.balance.desc()).show()

+--------------+-----------+-----------------+-------+
|account_number|customer_id|           branch|balance|
+--------------+-----------+-----------------+-------+
|          1010|         10|      Mumbai West| 200000|
|          1006|          6|   Hyderabad Main| 150000|
|          1002|          2|Bangalore Central| 120000|
|          1009|          9|Bangalore Central| 110000|
|          1004|          4|      Delhi North|  95000|
|          1001|          1|   Hyderabad Main|  85000|
|          1008|          8|      Delhi North|  70000|
|          1005|          5|    Chennai South|  60000|
|          1011|         11|   Hyderabad Main|  50000|
|          1003|          3|      Mumbai West|  45000|
|          1012|         12|      Delhi North|  40000|
|          1007|          7|        Pune East|  30000|
+--------------+-----------+-----------------+-------+



In [29]:
accounts.sort(accounts.balance.desc()).show(5)

+--------------+-----------+-----------------+-------+
|account_number|customer_id|           branch|balance|
+--------------+-----------+-----------------+-------+
|          1010|         10|      Mumbai West| 200000|
|          1006|          6|   Hyderabad Main| 150000|
|          1002|          2|Bangalore Central| 120000|
|          1009|          9|Bangalore Central| 110000|
|          1004|          4|      Delhi North|  95000|
+--------------+-----------+-----------------+-------+
only showing top 5 rows


In [30]:
accounts.sort(accounts.balance).show(5)

+--------------+-----------+--------------+-------+
|account_number|customer_id|        branch|balance|
+--------------+-----------+--------------+-------+
|          1007|          7|     Pune East|  30000|
|          1012|         12|   Delhi North|  40000|
|          1003|          3|   Mumbai West|  45000|
|          1011|         11|Hyderabad Main|  50000|
|          1005|          5| Chennai South|  60000|
+--------------+-----------+--------------+-------+
only showing top 5 rows


In [31]:
transactions.sort(transactions.amount.desc()).show()

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|            14|      1010|  Credit| 60000|2024-03-07|
|            13|      1006|  Credit| 50000|2024-03-07|
|             9|      1009|  Credit| 40000|2024-03-05|
|            10|      1010|   Debit| 35000|2024-03-05|
|             5|      1005|  Credit| 30000|2024-03-03|
|             1|      1001|  Credit| 25000|2024-03-01|
|            18|      1004|  Credit| 22000|2024-03-09|
|             6|      1006|   Debit| 20000|2024-03-03|
|            12|      1002|  Credit| 18000|2024-03-06|
|            16|      1012|  Credit| 16000|2024-03-08|
|             2|      1002|   Debit| 15000|2024-03-01|
|            20|      1009|   Debit| 14000|2024-03-10|
|             8|      1008|   Debit| 12000|2024-03-04|
|            19|      1005|   Debit| 11000|2024-03-10|
|             3|      1003|  Credit| 10000|2024-03-02|
|         

In [32]:
transactions.sort(transactions.amount.desc()).show(3)

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|            14|      1010|  Credit| 60000|2024-03-07|
|            13|      1006|  Credit| 50000|2024-03-07|
|             9|      1009|  Credit| 40000|2024-03-05|
+--------------+----------+--------+------+----------+
only showing top 3 rows


In [33]:
transactions.sort(transactions.txn_date).show()

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|             1|      1001|  Credit| 25000|2024-03-01|
|             2|      1002|   Debit| 15000|2024-03-01|
|             3|      1003|  Credit| 10000|2024-03-02|
|             4|      1004|   Debit|  5000|2024-03-02|
|             5|      1005|  Credit| 30000|2024-03-03|
|             6|      1006|   Debit| 20000|2024-03-03|
|             7|      1007|  Credit|  8000|2024-03-04|
|             8|      1008|   Debit| 12000|2024-03-04|
|             9|      1009|  Credit| 40000|2024-03-05|
|            10|      1010|   Debit| 35000|2024-03-05|
|            11|      1001|   Debit|  7000|2024-03-06|
|            12|      1002|  Credit| 18000|2024-03-06|
|            13|      1006|  Credit| 50000|2024-03-07|
|            14|      1010|  Credit| 60000|2024-03-07|
|            15|      1011|   Debit|  9000|2024-03-08|
|         

In [34]:
customers.sort(customers.city,customers.age).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
+-----------+------+---------+---+------------+-----------+



In [35]:
transactions.sort(transactions.txn_date.desc()).show(5)

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|            19|      1005|   Debit| 11000|2024-03-10|
|            20|      1009|   Debit| 14000|2024-03-10|
|            18|      1004|  Credit| 22000|2024-03-09|
|            17|      1003|   Debit|  4000|2024-03-09|
|            15|      1011|   Debit|  9000|2024-03-08|
+--------------+----------+--------+------+----------+
only showing top 5 rows


---

## Section 4 — Aggregations

31. Find total account balance.
32. Find average account balance.
33. Find highest account balance.
34. Find lowest account balance.
35. Count customers by city.
36. Count customers by account type.
37. Count transactions by transaction type.
38. Find total Credit transaction amount.
39. Find total Debit transaction amount.
40. Find average transaction amount.

---

In [60]:
from pyspark.sql.functions import sum,max,avg,min,count

In [42]:
accounts.show()

+--------------+-----------+-----------------+-------+
|account_number|customer_id|           branch|balance|
+--------------+-----------+-----------------+-------+
|          1001|          1|   Hyderabad Main|  85000|
|          1002|          2|Bangalore Central| 120000|
|          1003|          3|      Mumbai West|  45000|
|          1004|          4|      Delhi North|  95000|
|          1005|          5|    Chennai South|  60000|
|          1006|          6|   Hyderabad Main| 150000|
|          1007|          7|        Pune East|  30000|
|          1008|          8|      Delhi North|  70000|
|          1009|          9|Bangalore Central| 110000|
|          1010|         10|      Mumbai West| 200000|
|          1011|         11|   Hyderabad Main|  50000|
|          1012|         12|      Delhi North|  40000|
+--------------+-----------+-----------------+-------+



In [49]:
accounts.select(
    sum(accounts.balance).alias("TotalAccountBalance"),
    avg(accounts.balance).alias("AverageAccountBalance"),
    max(accounts.balance).alias("HighestAccountBalance"),
    min(accounts.balance).alias("LowestAccountBalance")
).show()

+-------------------+---------------------+---------------------+--------------------+
|TotalAccountBalance|AverageAccountBalance|HighestAccountBalance|LowestAccountBalance|
+-------------------+---------------------+---------------------+--------------------+
|            1055000|    87916.66666666667|               200000|               30000|
+-------------------+---------------------+---------------------+--------------------+



In [50]:
customers.groupBy(customers.city).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [51]:
customers.groupBy(customers.account_type).count().show()

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+



In [52]:
transactions.groupBy(transactions.txn_type).count().show()

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|   10|
|   Debit|   10|
+--------+-----+



In [54]:
transactions.filter(transactions.txn_type=="Credit").agg(sum(transactions.amount).alias("CreditTransactionAmount")).show()

+-----------------------+
|CreditTransactionAmount|
+-----------------------+
|                 279000|
+-----------------------+



In [55]:
transactions.filter(transactions.txn_type=="Debit").agg(sum(transactions.amount).alias("DebitTransactionAmount")).show()

+----------------------+
|DebitTransactionAmount|
+----------------------+
|                132000|
+----------------------+



In [56]:
transactions.select(
    avg(transactions.amount).alias("AverageTransactionAmount")
).show()

+------------------------+
|AverageTransactionAmount|
+------------------------+
|                 20550.0|
+------------------------+



---

## Section 5 — GroupBy Analytics

41. Find total balance by branch.
42. Find average balance by branch.
43. Count accounts by branch.
44. Find total transaction amount by account_id.
45. Find transaction count by account_id.
46. Find total transaction amount by txn_type.
47. Find transaction count by date.
48. Find average customer age by city.
49. Find total balance by account type after joining customers and accounts.
50. Find count of Savings and Current accounts by city.

---

In [65]:
accounts.groupBy(accounts.branch).agg(
    sum(accounts.balance).alias("TotalBalanceByBranch"),
    avg(accounts.balance).alias("AverageBalanceByBranch"),
    count(accounts.branch).alias("CountByBranch")).show()

+-----------------+--------------------+----------------------+-------------+
|           branch|TotalBalanceByBranch|AverageBalanceByBranch|CountByBranch|
+-----------------+--------------------+----------------------+-------------+
|      Delhi North|              205000|     68333.33333333333|            3|
|   Hyderabad Main|              285000|               95000.0|            3|
|        Pune East|               30000|               30000.0|            1|
|      Mumbai West|              245000|              122500.0|            2|
|    Chennai South|               60000|               60000.0|            1|
|Bangalore Central|              230000|              115000.0|            2|
+-----------------+--------------------+----------------------+-------------+



In [66]:
transactions.groupBy(transactions.account_id).agg(
    sum(transactions.amount).alias("TotalTransactionAmount"),
    count(transactions.account_id).alias("TransactionCount")).show()

+----------+----------------------+----------------+
|account_id|TotalTransactionAmount|TransactionCount|
+----------+----------------------+----------------+
|      1005|                 41000|               2|
|      1008|                 12000|               1|
|      1010|                 95000|               2|
|      1002|                 33000|               2|
|      1001|                 32000|               2|
|      1006|                 70000|               2|
|      1007|                  8000|               1|
|      1003|                 14000|               2|
|      1004|                 27000|               2|
|      1011|                  9000|               1|
|      1012|                 16000|               1|
|      1009|                 54000|               2|
+----------+----------------------+----------------+



In [67]:
transactions.groupBy(transactions.txn_type).agg(sum(transactions.amount).alias("TotalTransactionAmount")).show()

+--------+----------------------+
|txn_type|TotalTransactionAmount|
+--------+----------------------+
|  Credit|                279000|
|   Debit|                132000|
+--------+----------------------+



In [68]:
transactions.groupBy(transactions.txn_date).count().show()

+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-05|    2|
|2024-03-06|    2|
|2024-03-08|    2|
|2024-03-04|    2|
|2024-03-02|    2|
|2024-03-09|    2|
|2024-03-01|    2|
|2024-03-03|    2|
|2024-03-07|    2|
|2024-03-10|    2|
+----------+-----+



In [69]:
customers.groupBy(customers.city).agg(avg(customers.age).alias("AverageCustomerAge")).show()

+---------+------------------+
|     city|AverageCustomerAge|
+---------+------------------+
|Bangalore|              30.0|
|  Chennai|              30.0|
|   Mumbai|              34.5|
|     Pune|              38.0|
|    Delhi|28.666666666666668|
|Hyderabad|31.333333333333332|
+---------+------------------+



In [70]:
customers.join(accounts,customers.customer_id==accounts.customer_id,"inner").groupBy(accounts.branch).agg(sum(accounts.balance).alias("TotalBalanceByBranch")).show()

+-----------------+--------------------+
|           branch|TotalBalanceByBranch|
+-----------------+--------------------+
|      Delhi North|              205000|
|   Hyderabad Main|              285000|
|        Pune East|               30000|
|      Mumbai West|              245000|
|    Chennai South|               60000|
|Bangalore Central|              230000|
+-----------------+--------------------+



In [73]:
customers.groupBy(customers.city,customers.account_type).agg(count("*")).orderBy(customers.city).show()

+---------+------------+--------+
|     city|account_type|count(1)|
+---------+------------+--------+
|Bangalore|     Savings|       1|
|Bangalore|     Current|       1|
|  Chennai|     Current|       1|
|    Delhi|     Savings|       3|
|Hyderabad|     Savings|       3|
|   Mumbai|     Savings|       1|
|   Mumbai|     Current|       1|
|     Pune|     Current|       1|
+---------+------------+--------+



---

## Section 6 — Joins

51. Join customers with accounts using customer_id.
52. Show customer name, city, branch, and balance.
53. Join accounts with transactions using account_id.
54. Show account_id, txn_type, amount, and balance.
55. Join accounts with branches using branch.
56. Show branch, region, manager, and balance.
57. Join customers, accounts, and transactions.
58. Show customer name, city, txn_type, amount, and txn_date.
59. Join customers, accounts, branches, and transactions.
60. Find total transaction amount by customer name.

---

In [74]:
customers.join(accounts,customers.customer_id==accounts.customer_id).show()

+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_number|customer_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|          1001|          1|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|          1002|          2|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|          1003|          3|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|          1004|          4|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|          1005|          5|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|          1006|          6|   Hyderab

In [75]:
customers.join(accounts,customers.customer_id==accounts.customer_id).select(customers.name,customers.city,accounts.branch,accounts.balance).show()

+------+---------+-----------------+-------+
|  name|     city|           branch|balance|
+------+---------+-----------------+-------+
| Rahul|Hyderabad|   Hyderabad Main|  85000|
| Sneha|Bangalore|Bangalore Central| 120000|
| Arjun|   Mumbai|      Mumbai West|  45000|
| Priya|    Delhi|      Delhi North|  95000|
| Karan|  Chennai|    Chennai South|  60000|
| Meera|Hyderabad|   Hyderabad Main| 150000|
|  Amit|     Pune|        Pune East|  30000|
|  Neha|    Delhi|      Delhi North|  70000|
| Divya|Bangalore|Bangalore Central| 110000|
|Vikram|   Mumbai|      Mumbai West| 200000|
|Farhan|Hyderabad|   Hyderabad Main|  50000|
|Simran|    Delhi|      Delhi North|  40000|
+------+---------+-----------------+-------+



In [84]:
accounts.join(transactions, accounts.account_number == transactions.account_id).show()

+--------------+-----------+-----------------+-------+--------------+----------+--------+------+----------+
|account_number|customer_id|           branch|balance|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+-----------+-----------------+-------+--------------+----------+--------+------+----------+
|          1001|          1|   Hyderabad Main|  85000|             1|      1001|  Credit| 25000|2024-03-01|
|          1002|          2|Bangalore Central| 120000|             2|      1002|   Debit| 15000|2024-03-01|
|          1003|          3|      Mumbai West|  45000|             3|      1003|  Credit| 10000|2024-03-02|
|          1004|          4|      Delhi North|  95000|             4|      1004|   Debit|  5000|2024-03-02|
|          1005|          5|    Chennai South|  60000|             5|      1005|  Credit| 30000|2024-03-03|
|          1006|          6|   Hyderabad Main| 150000|             6|      1006|   Debit| 20000|2024-03-03|
|          1007|          7|

In [85]:
accounts.join(transactions, accounts.account_number == transactions.account_id).select(transactions.account_id,transactions.txn_type,accounts.balance,transactions.amount).show()

+----------+--------+-------+------+
|account_id|txn_type|balance|amount|
+----------+--------+-------+------+
|      1001|  Credit|  85000| 25000|
|      1002|   Debit| 120000| 15000|
|      1003|  Credit|  45000| 10000|
|      1004|   Debit|  95000|  5000|
|      1005|  Credit|  60000| 30000|
|      1006|   Debit| 150000| 20000|
|      1007|  Credit|  30000|  8000|
|      1008|   Debit|  70000| 12000|
|      1009|  Credit| 110000| 40000|
|      1010|   Debit| 200000| 35000|
|      1001|   Debit|  85000|  7000|
|      1002|  Credit| 120000| 18000|
|      1006|  Credit| 150000| 50000|
|      1010|  Credit| 200000| 60000|
|      1011|   Debit|  50000|  9000|
|      1012|  Credit|  40000| 16000|
|      1003|   Debit|  45000|  4000|
|      1004|  Credit|  95000| 22000|
|      1005|   Debit|  60000| 11000|
|      1009|   Debit| 110000| 14000|
+----------+--------+-------+------+



In [87]:
accounts.join(branches,accounts.branch==branches.branch).show()

+--------------+-----------+-----------------+-------+-----------------+------+-------+
|account_number|customer_id|           branch|balance|           branch|region|manager|
+--------------+-----------+-----------------+-------+-----------------+------+-------+
|          1001|          1|   Hyderabad Main|  85000|   Hyderabad Main| South| Ramesh|
|          1002|          2|Bangalore Central| 120000|Bangalore Central| South|  Leena|
|          1003|          3|      Mumbai West|  45000|      Mumbai West|  West| Joseph|
|          1004|          4|      Delhi North|  95000|      Delhi North| North|   Sara|
|          1005|          5|    Chennai South|  60000|    Chennai South| South|  Kumar|
|          1006|          6|   Hyderabad Main| 150000|   Hyderabad Main| South| Ramesh|
|          1007|          7|        Pune East|  30000|        Pune East|  West|  Anita|
|          1008|          8|      Delhi North|  70000|      Delhi North| North|   Sara|
|          1009|          9|Bang

In [88]:
accounts.join(branches,accounts.branch==branches.branch).select(branches.branch,branches.region,branches.manager,accounts.balance).show()

+-----------------+------+-------+-------+
|           branch|region|manager|balance|
+-----------------+------+-------+-------+
|   Hyderabad Main| South| Ramesh|  85000|
|Bangalore Central| South|  Leena| 120000|
|      Mumbai West|  West| Joseph|  45000|
|      Delhi North| North|   Sara|  95000|
|    Chennai South| South|  Kumar|  60000|
|   Hyderabad Main| South| Ramesh| 150000|
|        Pune East|  West|  Anita|  30000|
|      Delhi North| North|   Sara|  70000|
|Bangalore Central| South|  Leena| 110000|
|      Mumbai West|  West| Joseph| 200000|
|   Hyderabad Main| South| Ramesh|  50000|
|      Delhi North| North|   Sara|  40000|
+-----------------+------+-------+-------+



In [90]:
customers.join(accounts,customers.customer_id==accounts.customer_id).join(transactions, accounts.account_number == transactions.account_id).show()

+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+--------------+----------+--------+------+----------+
|customer_id|  name|     city|age|account_type|signup_date|account_number|customer_id|           branch|balance|transaction_id|account_id|txn_type|amount|  txn_date|
+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+--------------+----------+--------+------+----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|          1001|          1|   Hyderabad Main|  85000|            11|      1001|   Debit|  7000|2024-03-06|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|          1001|          1|   Hyderabad Main|  85000|             1|      1001|  Credit| 25000|2024-03-01|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|          1002|          2|Bangalore Central| 120000|            12|      1002|  Credit| 18000|2024-03-06|
|   

In [91]:
customers.join(accounts,customers.customer_id==accounts.customer_id).join(transactions, accounts.account_number == transactions.account_id).select(customers.name,customers.city,transactions.txn_type,transactions.amount,transactions.txn_date).show()

+------+---------+--------+------+----------+
|  name|     city|txn_type|amount|  txn_date|
+------+---------+--------+------+----------+
| Rahul|Hyderabad|   Debit|  7000|2024-03-06|
| Rahul|Hyderabad|  Credit| 25000|2024-03-01|
| Sneha|Bangalore|  Credit| 18000|2024-03-06|
| Sneha|Bangalore|   Debit| 15000|2024-03-01|
| Arjun|   Mumbai|   Debit|  4000|2024-03-09|
| Arjun|   Mumbai|  Credit| 10000|2024-03-02|
| Priya|    Delhi|  Credit| 22000|2024-03-09|
| Priya|    Delhi|   Debit|  5000|2024-03-02|
| Karan|  Chennai|   Debit| 11000|2024-03-10|
| Karan|  Chennai|  Credit| 30000|2024-03-03|
| Meera|Hyderabad|  Credit| 50000|2024-03-07|
| Meera|Hyderabad|   Debit| 20000|2024-03-03|
|  Amit|     Pune|  Credit|  8000|2024-03-04|
|  Neha|    Delhi|   Debit| 12000|2024-03-04|
| Divya|Bangalore|   Debit| 14000|2024-03-10|
| Divya|Bangalore|  Credit| 40000|2024-03-05|
|Vikram|   Mumbai|  Credit| 60000|2024-03-07|
|Vikram|   Mumbai|   Debit| 35000|2024-03-05|
|Farhan|Hyderabad|   Debit|  9000|

In [92]:
customers.join(accounts,customers.customer_id==accounts.customer_id).join(transactions, accounts.account_number == transactions.account_id).join(branches,accounts.branch==branches.branch).show()

+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+--------------+----------+--------+------+----------+-----------------+------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_number|customer_id|           branch|balance|transaction_id|account_id|txn_type|amount|  txn_date|           branch|region|manager|
+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+--------------+----------+--------+------+----------+-----------------+------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|          1001|          1|   Hyderabad Main|  85000|            11|      1001|   Debit|  7000|2024-03-06|   Hyderabad Main| South| Ramesh|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|          1001|          1|   Hyderabad Main|  85000|             1|      1001|  Credit| 25000|2024-03-01|   Hyderabad Main| South| Ramesh|
|    

In [94]:
customers.join(accounts,customers.customer_id==accounts.customer_id).join(transactions, accounts.account_number == transactions.account_id).join(branches,accounts.branch==branches.branch).groupBy(customers.name).agg(sum(transactions.amount).alias("TransactionAmount")).show()

+------+-----------------+
|  name|TransactionAmount|
+------+-----------------+
| Divya|            54000|
| Meera|            70000|
| Sneha|            33000|
| Priya|            27000|
|Vikram|            95000|
|Simran|            16000|
| Rahul|            32000|
| Arjun|            14000|
|  Amit|             8000|
|  Neha|            12000|
|Farhan|             9000|
| Karan|            41000|
+------+-----------------+



---

## Section 7 — Updating Data with withColumn

61. Add balance_in_lakhs = balance / 100000.
62. Add bank_name = "BotCampus Bank".
63. Add annual_service_fee = balance * 0.01.
64. Add net_balance = balance - annual_service_fee.
65. Add is_high_balance where balance > 100000.
66. Add txn_amount_in_k = amount / 1000.
67. Add country = "India" to customers.
68. Add customer_label = name + " - " + city.
69. Add branch_label = branch + " - " + region.
70. Add risk_flag for transactions above 40000.

---

In [101]:
customers.show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [105]:
from pyspark.sql.functions import *

In [111]:
accounts.withColumns(
    {"balance_in_lakhs":accounts.balance/100000,
    "bank_name":lit("BotCampus Bank"),
    "annual_service_fee":accounts.balance*0.01,
    "net_balance":accounts.balance-(accounts.balance*0.01),
    "is_high_balance":when(accounts.balance>100000,"Yes").otherwise("No")
     }).show()

+--------------+-----------+-----------------+-------+----------------+--------------+------------------+-----------+---------------+
|account_number|customer_id|           branch|balance|balance_in_lakhs|     bank_name|annual_service_fee|net_balance|is_high_balance|
+--------------+-----------+-----------------+-------+----------------+--------------+------------------+-----------+---------------+
|          1001|          1|   Hyderabad Main|  85000|            0.85|BotCampus Bank|             850.0|    84150.0|             No|
|          1002|          2|Bangalore Central| 120000|             1.2|BotCampus Bank|            1200.0|   118800.0|            Yes|
|          1003|          3|      Mumbai West|  45000|            0.45|BotCampus Bank|             450.0|    44550.0|             No|
|          1004|          4|      Delhi North|  95000|            0.95|BotCampus Bank|             950.0|    94050.0|             No|
|          1005|          5|    Chennai South|  60000|        

In [112]:
transactions.withColumn("txn_amount_in_k",transactions.amount/1000).show()

+--------------+----------+--------+------+----------+---------------+
|transaction_id|account_id|txn_type|amount|  txn_date|txn_amount_in_k|
+--------------+----------+--------+------+----------+---------------+
|             1|      1001|  Credit| 25000|2024-03-01|           25.0|
|             2|      1002|   Debit| 15000|2024-03-01|           15.0|
|             3|      1003|  Credit| 10000|2024-03-02|           10.0|
|             4|      1004|   Debit|  5000|2024-03-02|            5.0|
|             5|      1005|  Credit| 30000|2024-03-03|           30.0|
|             6|      1006|   Debit| 20000|2024-03-03|           20.0|
|             7|      1007|  Credit|  8000|2024-03-04|            8.0|
|             8|      1008|   Debit| 12000|2024-03-04|           12.0|
|             9|      1009|  Credit| 40000|2024-03-05|           40.0|
|            10|      1010|   Debit| 35000|2024-03-05|           35.0|
|            11|      1001|   Debit|  7000|2024-03-06|            7.0|
|     

In [115]:
customers.withColumn("customer_label",concat(customers.name,lit(" - "),customers.city)).show()

+-----------+------+---------+---+------------+-----------+------------------+
|customer_id|  name|     city|age|account_type|signup_date|    customer_label|
+-----------+------+---------+---+------------+-----------+------------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10| Rahul - Hyderabad|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| Sneha - Bangalore|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Arjun - Mumbai|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|     Priya - Delhi|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|   Karan - Chennai|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10| Meera - Hyderabad|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|       Amit - Pune|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      Neha - Delhi|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15| Divya - Bangalore|
|         10|Vikram|   Mumbai| 42|     Current| 2023

In [117]:
branches.withColumn("branch_label",concat(branches.branch,lit(" - "),branches.region)).show()

+-----------------+------+-------+--------------------+
|           branch|region|manager|        branch_label|
+-----------------+------+-------+--------------------+
|   Hyderabad Main| South| Ramesh|Hyderabad Main - ...|
|Bangalore Central| South|  Leena|Bangalore Central...|
|      Mumbai West|  West| Joseph|  Mumbai West - West|
|      Delhi North| North|   Sara| Delhi North - North|
|    Chennai South| South|  Kumar|Chennai South - S...|
|        Pune East|  West|  Anita|    Pune East - West|
+-----------------+------+-------+--------------------+



In [118]:
transactions.withColumn("risk_flag",when(transactions.amount>40000,"yes").otherwise("no")).show()

+--------------+----------+--------+------+----------+---------+
|transaction_id|account_id|txn_type|amount|  txn_date|risk_flag|
+--------------+----------+--------+------+----------+---------+
|             1|      1001|  Credit| 25000|2024-03-01|       no|
|             2|      1002|   Debit| 15000|2024-03-01|       no|
|             3|      1003|  Credit| 10000|2024-03-02|       no|
|             4|      1004|   Debit|  5000|2024-03-02|       no|
|             5|      1005|  Credit| 30000|2024-03-03|       no|
|             6|      1006|   Debit| 20000|2024-03-03|       no|
|             7|      1007|  Credit|  8000|2024-03-04|       no|
|             8|      1008|   Debit| 12000|2024-03-04|       no|
|             9|      1009|  Credit| 40000|2024-03-05|       no|
|            10|      1010|   Debit| 35000|2024-03-05|       no|
|            11|      1001|   Debit|  7000|2024-03-06|       no|
|            12|      1002|  Credit| 18000|2024-03-06|       no|
|            13|      100

---

## Section 8 — when / otherwise

71. Create balance category: High if balance >= 100000, Medium if >= 50000, else Low.
72. Create age group: Young if age < 30, Adult if age < 40, else Senior.
73. Create transaction category: Large if amount >= 30000, Medium if >= 10000, else Small.
74. Create region priority: South = High Priority, North = Medium Priority, West = Watch.
75. Create account type label: Savings = Retail, Current = Business.

---


In [120]:
accounts.withColumn("balance_category",when(accounts.balance>=100000,"High").when(accounts.balance>=50000,"Medium").otherwise("Low")).show()

+--------------+-----------+-----------------+-------+----------------+
|account_number|customer_id|           branch|balance|balance_category|
+--------------+-----------+-----------------+-------+----------------+
|          1001|          1|   Hyderabad Main|  85000|          Medium|
|          1002|          2|Bangalore Central| 120000|            High|
|          1003|          3|      Mumbai West|  45000|             Low|
|          1004|          4|      Delhi North|  95000|          Medium|
|          1005|          5|    Chennai South|  60000|          Medium|
|          1006|          6|   Hyderabad Main| 150000|            High|
|          1007|          7|        Pune East|  30000|             Low|
|          1008|          8|      Delhi North|  70000|          Medium|
|          1009|          9|Bangalore Central| 110000|            High|
|          1010|         10|      Mumbai West| 200000|            High|
|          1011|         11|   Hyderabad Main|  50000|          

In [121]:
customers.withColumn("age_group",when(customers.age<30,"Young").when(customers.age<40,"Adult").otherwise("Senior")).show()

+-----------+------+---------+---+------------+-----------+---------+
|customer_id|  name|     city|age|account_type|signup_date|age_group|
+-----------+------+---------+---+------------+-----------+---------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|    Young|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|    Adult|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Young|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|    Adult|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|    Adult|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|    Adult|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|    Adult|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|    Young|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|    Young|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|   Senior|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|    Adult|
|         12|Simran|

In [122]:
transactions.withColumn("transaction_category",when(transactions.amount>=30000,"Large").when(transactions.amount>=10000,"Medium").otherwise("Small")).show()

+--------------+----------+--------+------+----------+--------------------+
|transaction_id|account_id|txn_type|amount|  txn_date|transaction_category|
+--------------+----------+--------+------+----------+--------------------+
|             1|      1001|  Credit| 25000|2024-03-01|              Medium|
|             2|      1002|   Debit| 15000|2024-03-01|              Medium|
|             3|      1003|  Credit| 10000|2024-03-02|              Medium|
|             4|      1004|   Debit|  5000|2024-03-02|               Small|
|             5|      1005|  Credit| 30000|2024-03-03|               Large|
|             6|      1006|   Debit| 20000|2024-03-03|              Medium|
|             7|      1007|  Credit|  8000|2024-03-04|               Small|
|             8|      1008|   Debit| 12000|2024-03-04|              Medium|
|             9|      1009|  Credit| 40000|2024-03-05|               Large|
|            10|      1010|   Debit| 35000|2024-03-05|               Large|
|           

In [123]:
branches.withColumn("region_priority",when(branches.region=="South","High Priority").when(branches.region=="North","Medium Priority").otherwise("Watch")).show()

+-----------------+------+-------+---------------+
|           branch|region|manager|region_priority|
+-----------------+------+-------+---------------+
|   Hyderabad Main| South| Ramesh|  High Priority|
|Bangalore Central| South|  Leena|  High Priority|
|      Mumbai West|  West| Joseph|          Watch|
|      Delhi North| North|   Sara|Medium Priority|
|    Chennai South| South|  Kumar|  High Priority|
|        Pune East|  West|  Anita|          Watch|
+-----------------+------+-------+---------------+



In [124]:
customers.withColumn("account_type_label",when(customers.account_type=="Savings","Retail").otherwise("Business")).show()

+-----------+------+---------+---+------------+-----------+------------------+
|customer_id|  name|     city|age|account_type|signup_date|account_type_label|
+-----------+------+---------+---+------------+-----------+------------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|            Retail|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|          Business|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|            Retail|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|            Retail|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|          Business|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|            Retail|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|          Business|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|            Retail|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|            Retail|
|         10|Vikram|   Mumbai| 42|     Current| 2023

---

## Section 9 — Date Functions

76. Convert signup_date to date type.
77. Extract signup year.
78. Extract signup month.
79. Convert txn_date to date type.
80. Extract transaction month.
81. Count transactions by transaction date.
82. Count transactions by month.
83. Find customers signed up after 2023-06-01.
84. Find days since signup.
85. Find days since transaction.

---

In [126]:
customers.withColumn("signup_date",to_date(customers.signup_date)).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [128]:
customers.select(
    year(customers.signup_date).alias("signup_year"),
    month(customers.signup_date).alias("signup_month")
).show()

+-----------+------------+
|signup_year|signup_month|
+-----------+------------+
|       2023|           1|
|       2023|           2|
|       2023|           3|
|       2023|           4|
|       2023|           5|
|       2023|           6|
|       2023|           6|
|       2023|           7|
|       2023|           7|
|       2023|           8|
|       2023|           8|
|       2023|           8|
+-----------+------------+



In [129]:
transactions.withColumn("txn_date",to_date(transactions.txn_date)).show()

+--------------+----------+--------+------+----------+
|transaction_id|account_id|txn_type|amount|  txn_date|
+--------------+----------+--------+------+----------+
|             1|      1001|  Credit| 25000|2024-03-01|
|             2|      1002|   Debit| 15000|2024-03-01|
|             3|      1003|  Credit| 10000|2024-03-02|
|             4|      1004|   Debit|  5000|2024-03-02|
|             5|      1005|  Credit| 30000|2024-03-03|
|             6|      1006|   Debit| 20000|2024-03-03|
|             7|      1007|  Credit|  8000|2024-03-04|
|             8|      1008|   Debit| 12000|2024-03-04|
|             9|      1009|  Credit| 40000|2024-03-05|
|            10|      1010|   Debit| 35000|2024-03-05|
|            11|      1001|   Debit|  7000|2024-03-06|
|            12|      1002|  Credit| 18000|2024-03-06|
|            13|      1006|  Credit| 50000|2024-03-07|
|            14|      1010|  Credit| 60000|2024-03-07|
|            15|      1011|   Debit|  9000|2024-03-08|
|         

In [130]:
transactions.select(
    month(transactions.txn_date).alias("TransactionMonth")
).show()

+----------------+
|TransactionMonth|
+----------------+
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
+----------------+



In [135]:
transactions.select(
    month(transactions.txn_date).alias("TransactionMonth")
).groupBy("TransactionMonth").agg(count("TransactionMonth").alias("TransactionsCount")).show()

+----------------+-----------------+
|TransactionMonth|TransactionsCount|
+----------------+-----------------+
|               3|               20|
+----------------+-----------------+



In [134]:
transactions.groupBy(transactions.txn_date).agg(count(transactions.txn_date).alias("TransactionsCount")).show()

+----------+-----------------+
|  txn_date|TransactionsCount|
+----------+-----------------+
|2024-03-05|                2|
|2024-03-06|                2|
|2024-03-08|                2|
|2024-03-04|                2|
|2024-03-02|                2|
|2024-03-09|                2|
|2024-03-01|                2|
|2024-03-03|                2|
|2024-03-07|                2|
|2024-03-10|                2|
+----------+-----------------+



In [136]:
customers.filter(customers.signup_date>"2023-06-01").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [139]:
customers.select(datediff(current_date(),customers.signup_date).alias("DaysSinceSignUp")).show()

+---------------+
|DaysSinceSignUp|
+---------------+
|           1203|
|           1170|
|           1140|
|           1108|
|           1082|
|           1052|
|           1040|
|           1022|
|           1017|
|           1000|
|            991|
|            980|
+---------------+



In [141]:
transactions.select(datediff(current_date(),transactions.txn_date).alias("DaysSinceTransaction")).show()

+--------------------+
|DaysSinceTransaction|
+--------------------+
|                 787|
|                 787|
|                 786|
|                 786|
|                 785|
|                 785|
|                 784|
|                 784|
|                 783|
|                 783|
|                 782|
|                 782|
|                 781|
|                 781|
|                 780|
|                 780|
|                 779|
|                 779|
|                 778|
|                 778|
+--------------------+



---

## Section 10 — Window Functions

86. Rank customers by balance within city.
87. Use row_number to find top balance account per city.
88. Use dense_rank to rank transaction amount by transaction type.
89. Find top 2 transactions per account.
90. Create running transaction total per account_id.
91. Rank branches by total balance.
92. Rank cities by total customer balance.
93. Rank customers by total transaction amount.
94. Find highest transaction customer per city.
95. Find highest balance customer per region.

---


In [142]:
from pyspark.sql import Window

In [145]:
w=Window.partitionBy(customers.city).orderBy(accounts.balance.desc())

In [147]:
customers.join(accounts,customers.customer_id==accounts.customer_id).withColumn("customer_rank",rank().over(w)).show()

+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+-------------+
|customer_id|  name|     city|age|account_type|signup_date|account_number|customer_id|           branch|balance|customer_ranm|
+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+-------------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|          1002|          2|Bangalore Central| 120000|            1|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|          1009|          9|Bangalore Central| 110000|            2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|          1005|          5|    Chennai South|  60000|            1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|          1004|          4|      Delhi North|  95000|            1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|          1008|          8|      Delhi North|  70000

In [148]:
customers.join(accounts,customers.customer_id==accounts.customer_id).withColumn("customer_row_num",row_number().over(w)).show()

+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+----------------+
|customer_id|  name|     city|age|account_type|signup_date|account_number|customer_id|           branch|balance|customer_row_num|
+-----------+------+---------+---+------------+-----------+--------------+-----------+-----------------+-------+----------------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|          1002|          2|Bangalore Central| 120000|               1|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|          1009|          9|Bangalore Central| 110000|               2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|          1005|          5|    Chennai South|  60000|               1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|          1004|          4|      Delhi North|  95000|               1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|          1008|          8|    

In [153]:
w=Window.partitionBy(transactions.txn_type).orderBy(transactions.amount.desc())

In [150]:
transactions.withColumn("transaction_rank",dense_rank().over(w)).show()

+--------------+----------+--------+------+----------+----------------+
|transaction_id|account_id|txn_type|amount|  txn_date|transaction_rank|
+--------------+----------+--------+------+----------+----------------+
|            14|      1010|  Credit| 60000|2024-03-07|               1|
|            13|      1006|  Credit| 50000|2024-03-07|               2|
|             9|      1009|  Credit| 40000|2024-03-05|               3|
|             5|      1005|  Credit| 30000|2024-03-03|               4|
|             1|      1001|  Credit| 25000|2024-03-01|               5|
|            18|      1004|  Credit| 22000|2024-03-09|               6|
|            12|      1002|  Credit| 18000|2024-03-06|               7|
|            16|      1012|  Credit| 16000|2024-03-08|               8|
|             3|      1003|  Credit| 10000|2024-03-02|               9|
|             7|      1007|  Credit|  8000|2024-03-04|              10|
|            10|      1010|   Debit| 35000|2024-03-05|          

In [151]:
w=Window.orderBy(transactions.amount.desc())

In [152]:
transactions.withColumn("transaction_rank",rank().over(w)).show(2)

+--------------+----------+--------+------+----------+----------------+
|transaction_id|account_id|txn_type|amount|  txn_date|transaction_rank|
+--------------+----------+--------+------+----------+----------------+
|            14|      1010|  Credit| 60000|2024-03-07|               1|
|            13|      1006|  Credit| 50000|2024-03-07|               2|
+--------------+----------+--------+------+----------+----------------+
only showing top 2 rows


In [158]:
w=Window.partitionBy(transactions.account_id).orderBy(transactions.amount)

In [156]:
transactions.withColumn("running_total",sum(transactions.amount).over(w)).show()

+--------------+----------+--------+------+----------+-------------+
|transaction_id|account_id|txn_type|amount|  txn_date|running_total|
+--------------+----------+--------+------+----------+-------------+
|             1|      1001|  Credit| 25000|2024-03-01|        25000|
|            11|      1001|   Debit|  7000|2024-03-06|        32000|
|            12|      1002|  Credit| 18000|2024-03-06|        18000|
|             2|      1002|   Debit| 15000|2024-03-01|        33000|
|             3|      1003|  Credit| 10000|2024-03-02|        10000|
|            17|      1003|   Debit|  4000|2024-03-09|        14000|
|            18|      1004|  Credit| 22000|2024-03-09|        22000|
|             4|      1004|   Debit|  5000|2024-03-02|        27000|
|             5|      1005|  Credit| 30000|2024-03-03|        30000|
|            19|      1005|   Debit| 11000|2024-03-10|        41000|
|            13|      1006|  Credit| 50000|2024-03-07|        50000|
|             6|      1006|   Debi

In [159]:
w = Window.orderBy(col("total_balance").desc())

accounts.groupBy("branch").sum("balance").withColumnRenamed("sum(balance)", "total_balance").withColumn("rank", rank().over(w)).show()

+-----------------+-------------+----+
|           branch|total_balance|rank|
+-----------------+-------------+----+
|   Hyderabad Main|       285000|   1|
|      Mumbai West|       245000|   2|
|Bangalore Central|       230000|   3|
|      Delhi North|       205000|   4|
|    Chennai South|        60000|   5|
|        Pune East|        30000|   6|
+-----------------+-------------+----+



In [160]:
w = Window.orderBy(col("total_balance").desc())

customers.join(accounts, "customer_id").groupBy("city").sum("balance").withColumnRenamed("sum(balance)", "total_balance").withColumn("rank", rank().over(w)).show()

+---------+-------------+----+
|     city|total_balance|rank|
+---------+-------------+----+
|Hyderabad|       285000|   1|
|   Mumbai|       245000|   2|
|Bangalore|       230000|   3|
|    Delhi|       205000|   4|
|  Chennai|        60000|   5|
|     Pune|        30000|   6|
+---------+-------------+----+



In [161]:
w = Window.orderBy(col("total_amount").desc())

transactions.groupBy("account_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount").withColumn("rank", rank().over(w)).show()

+----------+------------+----+
|account_id|total_amount|rank|
+----------+------------+----+
|      1010|       95000|   1|
|      1006|       70000|   2|
|      1009|       54000|   3|
|      1005|       41000|   4|
|      1002|       33000|   5|
|      1001|       32000|   6|
|      1004|       27000|   7|
|      1012|       16000|   8|
|      1003|       14000|   9|
|      1008|       12000|  10|
|      1011|        9000|  11|
|      1007|        8000|  12|
+----------+------------+----+



In [163]:
w = Window.partitionBy("city").orderBy(col("amount").desc())

customers.join(accounts, "customer_id").join(transactions, accounts.account_number==transactions.account_id).withColumn("rn", row_number().over(w)).filter("rn = 1").show()

+-----------+------+---------+---+------------+-----------+--------------+-----------------+-------+--------------+----------+--------+------+----------+---+
|customer_id|  name|     city|age|account_type|signup_date|account_number|           branch|balance|transaction_id|account_id|txn_type|amount|  txn_date| rn|
+-----------+------+---------+---+------------+-----------+--------------+-----------------+-------+--------------+----------+--------+------+----------+---+
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|          1009|Bangalore Central| 110000|             9|      1009|  Credit| 40000|2024-03-05|  1|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|          1005|    Chennai South|  60000|             5|      1005|  Credit| 30000|2024-03-03|  1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|          1004|      Delhi North|  95000|            18|      1004|  Credit| 22000|2024-03-09|  1|
|          6| Meera|Hyderabad| 31|     Savings| 2023

In [164]:
w = Window.partitionBy("region").orderBy(col("balance").desc())

customers.join(accounts, "customer_id").join(branches, "branch").withColumn("rn", row_number().over(w)).filter("rn = 1").show()

+--------------+-----------+------+---------+---+------------+-----------+--------------+-------+------+-------+---+
|        branch|customer_id|  name|     city|age|account_type|signup_date|account_number|balance|region|manager| rn|
+--------------+-----------+------+---------+---+------------+-----------+--------------+-------+------+-------+---+
|   Delhi North|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|          1004|  95000| North|   Sara|  1|
|Hyderabad Main|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|          1006| 150000| South| Ramesh|  1|
|   Mumbai West|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|          1010| 200000|  West| Joseph|  1|
+--------------+-----------+------+---------+---+------------+-----------+--------------+-------+------+-------+---+



---

## Section 11 — Complex JSON

96. Read customer_profiles.json.
97. Extract customer_id, name, email, and phone.
98. Explode services.
99. Find customers using UPI.
100. Count how many customers use each service.

---

In [166]:
profiles = spark.read.json("customer_profiles.json", multiLine=True)

In [168]:
profiles.show()

+--------------------+-----------+------+--------------------+
|             contact|customer_id|  name|            services|
+--------------------+-----------+------+--------------------+
|{rahul@mail.com, ...|          1| Rahul|[UPI, Credit Card...|
|{sneha@mail.com, ...|          2| Sneha|   [UPI, Debit Card]|
|{arjun@mail.com, ...|          3| Arjun| [Net Banking, Loan]|
|{meera@mail.com, ...|          6| Meera|[UPI, Credit Card...|
|{vikram@mail.com,...|         10|Vikram|[Net Banking, Wea...|
+--------------------+-----------+------+--------------------+



In [169]:
profiles.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [170]:
profiles.select(profiles.customer_id,profiles.name,profiles.contact["email"],profiles.contact["phone"]).show()

+-----------+------+---------------+-------------+
|customer_id|  name|  contact.email|contact.phone|
+-----------+------+---------------+-------------+
|          1| Rahul| rahul@mail.com|   9000011111|
|          2| Sneha| sneha@mail.com|   9000022222|
|          3| Arjun| arjun@mail.com|   9000033333|
|          6| Meera| meera@mail.com|   9000066666|
|         10|Vikram|vikram@mail.com|   9000101010|
+-----------+------+---------------+-------------+



In [172]:
profiles.select(explode(profiles.services).alias("Services")).show()

+-----------+
|   Services|
+-----------+
|        UPI|
|Credit Card|
|Net Banking|
|        UPI|
| Debit Card|
|Net Banking|
|       Loan|
|        UPI|
|Credit Card|
|       Loan|
|Net Banking|
|     Wealth|
+-----------+



In [178]:
profiles.select(profiles.name,profiles.customer_id,explode(profiles.services).alias("Services")).filter("Services='UPI'").select(profiles.customer_id,profiles.name).show()

+-----------+-----+
|customer_id| name|
+-----------+-----+
|          1|Rahul|
|          2|Sneha|
|          6|Meera|
+-----------+-----+



In [180]:
profiles.withColumn("service", explode("services")).groupBy("service").agg(count("*").alias("customer_count")).show()

+-----------+--------------+
|    service|customer_count|
+-----------+--------------+
|     Wealth|             1|
|       Loan|             2|
|Credit Card|             2|
|Net Banking|             3|
| Debit Card|             1|
|        UPI|             3|
+-----------+--------------+

